In [1]:
import pandas as pd
import numpy as np
from google.oauth2 import service_account
from google.cloud import bigquery

# =========================================================
# CONFIG
# =========================================================
CREDS = "converge-database-4b8cc11a5506.json"
PROJECT_ID = "converge-database"
CUT_OFF_MONTH = "202602"

TARGET_TABLES = {
    "DENALI": "All_Contracts_FIA.All_Contracts_DENALI",
    "TETON": "All_Contracts_FIA.All_Contracts_TETON"
}

TARGET_FINAL_COLS = [
    "policy_number",
    "product",
    "term",
    "issue_year",
    "issue_month",
    "issue_day",
    "issue_date",
    "issue_state",
    "initial_premium",
    "initial_commission",
    "rate_version",
    "gender",
    "quota_share",
    "tax_status",
    "surrender_date",
    "termination_type",
    "annuitant_issue_age",
    "annuitant_gender"
]

REQUIRED_COLS = [
    "policy_number",
    "product",
    "term",
    "issue_year",
    "issue_month",
    "issue_day",
    "issue_date"
]

POLICY_COMPARE_COLS = [
    "term_final",
    "issueyear",
    "issuemonth",
    "issueday",
    "issue_date",
    "issue_state_final",
    "rateversion",
    "taxqualstatus",
    "ownergender",
    "joint_annuitant_gender",
    "joint_annuitant_issue_age"
]

client = bigquery.Client.from_service_account_json(
    json_credentials_path=CREDS
)

bq_credentials = service_account.Credentials.from_service_account_file(CREDS)

# =========================================================
# HELPERS
# =========================================================
def query_to_df(query):
    df = client.query(query).to_dataframe()
    df.columns = [c.strip().lower() for c in df.columns]

    # remove timezone from BigQuery datetime columns
    for c in df.columns:
        if pd.api.types.is_datetime64tz_dtype(df[c]):
            df[c] = df[c].dt.tz_localize(None)

    return df

def upload_to_gbq(df, target_table, if_exists="replace"):
    df.to_gbq(
        destination_table=target_table,
        project_id=PROJECT_ID,
        if_exists=if_exists,
        credentials=bq_credentials
    )
    print(f"Loaded {len(df)} rows to {target_table}")

def safe_numeric(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def safe_datetime(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
            if pd.api.types.is_datetime64tz_dtype(df[c]):
                df[c] = df[c].dt.tz_localize(None)
    return df

def add_missing_cols(df, cols_with_default):
    df = df.copy()
    for c, default_val in cols_with_default.items():
        if c not in df.columns:
            df[c] = default_val
    return df

def keep_first_by_key(df, subset_cols, sort_cols=None):
    df = df.copy()
    if sort_cols is not None:
        existing_sort_cols = [c for c in sort_cols if c in df.columns]
        if existing_sort_cols:
            df = df.sort_values(existing_sort_cols)
    return df.drop_duplicates(subset=subset_cols, keep="first")

def make_issue_date(df, year_col="issueyear", month_col="issuemonth", day_col="issueday", new_col="issue_date"):
    df = df.copy()

    if year_col not in df.columns or month_col not in df.columns or day_col not in df.columns:
        df[new_col] = pd.NaT
        return df

    df = safe_numeric(df, [year_col, month_col, day_col])

    date_str = (
        df[year_col].fillna(0).astype("Int64").astype(str).str.zfill(4) +
        df[month_col].fillna(0).astype("Int64").astype(str).str.zfill(2) +
        df[day_col].fillna(0).astype("Int64").astype(str).str.zfill(2)
    )

    df[new_col] = pd.to_datetime(date_str, format="%Y%m%d", errors="coerce")
    return df

def normalize_for_compare(series):
    s = series.copy()

    if pd.api.types.is_datetime64_any_dtype(s):
        return s.dt.strftime("%Y-%m-%d").fillna("<<EMPTY>>")

    if pd.api.types.is_numeric_dtype(s):
        return s.apply(lambda x: "<<EMPTY>>" if pd.isna(x) or x == 0 else str(x))

    return (
        s.astype("string")
         .str.strip()
         .replace({"": pd.NA, "0": pd.NA})
         .fillna("<<EMPTY>>")
    )

def derive_product_from_policy(policy_number):
    if pd.isna(policy_number):
        return pd.NA

    p = str(policy_number).strip().upper()

    if p.startswith("DB"):
        return "Denali Bonus"
    if p.startswith("D"):
        return "Denali"
    if p.startswith("TB"):
        return "Teton Bonus"
    if p.startswith("T"):
        return "Teton"

    return pd.NA

def assign_product_group(product):
    if pd.isna(product):
        return pd.NA

    p = str(product).strip().lower()

    if "denali" in p:
        return "DENALI"
    if "teton" in p:
        return "TETON"

    return pd.NA

def remove_timezone_for_excel(df):
    df = df.copy()

    for c in df.columns:
        if pd.api.types.is_datetime64tz_dtype(df[c]):
            df[c] = df[c].dt.tz_localize(None)
        elif df[c].dtype == "object":
            sample = df[c].dropna()
            if not sample.empty:
                first_val = sample.iloc[0]
                if isinstance(first_val, pd.Timestamp) and first_val.tz is not None:
                    df[c] = df[c].apply(
                        lambda x: x.tz_localize(None)
                        if isinstance(x, pd.Timestamp) and x.tz is not None
                        else x
                    )
    return df

def enforce_target_dtypes(df):
    df = df.copy()

    # integer fields expected in target
    int_cols = [
        "term",
        "issue_year",
        "issue_month",
        "issue_day",
        "rate_version",
        "annuitant_issue_age"
    ]

    for c in int_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    # date fields expected in target
    date_cols = ["issue_date", "surrender_date"]
    for c in date_cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.date

    return df

def write_exception_workbook(sheets_dict, file_name):
    with pd.ExcelWriter(file_name, engine="openpyxl") as writer:
        for sheet_name, df in sheets_dict.items():
            clean_df = remove_timezone_for_excel(df)
            clean_df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    print(f"Excel workbook created: {file_name}")

# =========================================================
# SOURCE PULL
# =========================================================
def pull_source_tables(cutoff_month):
    queries = {
        "policy": f"""
            SELECT *
            FROM `converge-database.denali.policy`
            WHERE set_month < '{cutoff_month}'
        """,
        "seriatim_values": f"""
            SELECT *
            FROM `converge-database.denali.seriatim_values`
            WHERE set_month < '{cutoff_month}'
        """,
        "premiums": """
            SELECT *
            FROM `converge-database.denali.premiums`
        """,
        "commission": """
            SELECT *
            FROM `converge-database.denali.commissions`
        """
    }

    out = {}
    for name, q in queries.items():
        df = query_to_df(q)
        out[name] = df
        print(f"{name}: {len(df)} rows")
    return out

# =========================================================
# POLICY HISTORY CHECK
# =========================================================
def build_policy_history_and_exceptions(policy_df, cutoff_month):
    policy_df = policy_df.copy()

    policy_df = make_issue_date(policy_df, "issueyear", "issuemonth", "issueday", "issue_date")
    policy_df = safe_numeric(
        policy_df,
        ["plan", "issueyear", "issuemonth", "issueday", "issueage", "rateversion", "joint_annuitant_issue_age"]
    )

    policy_df = add_missing_cols(policy_df, {
        "set_month": None,
        "policynumber": None,
        "plan": pd.NA,
        "issueyear": pd.NA,
        "issuemonth": pd.NA,
        "issueday": pd.NA,
        "issue_date": pd.NaT,
        "stateofsale": None,
        "issueage": pd.NA,
        "rateversion": pd.NA,
        "taxqualstatus": None,
        "ownergender": None,
        "joint_annuitant_gender": None,
        "joint_annuitant_issue_age": pd.NA
    })

    policy_df["term_final"] = pd.to_numeric(policy_df["plan"], errors="coerce").astype("Int64").astype("string")
    policy_df["issue_state_final"] = policy_df["stateofsale"]

    policy_hist = policy_df[[
        "set_month",
        "policynumber",
        "term_final",
        "issueyear",
        "issuemonth",
        "issueday",
        "issue_date",
        "issue_state_final",
        "rateversion",
        "taxqualstatus",
        "ownergender",
        "joint_annuitant_gender",
        "joint_annuitant_issue_age"
    ]].copy()

    policy_hist = policy_hist.sort_values(["policynumber", "set_month"])

    exception_rows = []

    for policy_number, grp in policy_hist.groupby("policynumber", dropna=False):
        if pd.isna(policy_number):
            continue

        grp = grp.copy()

        for col in POLICY_COMPARE_COLS:
            norm_vals = normalize_for_compare(grp[col])
            unique_vals = pd.Series(norm_vals.unique())
            unique_vals = unique_vals[unique_vals != "<<EMPTY>>"]

            if len(unique_vals) > 1:
                base_row = grp.iloc[0]
                base_val = base_row[col]

                for _, row in grp.iloc[1:].iterrows():
                    row_val_norm = normalize_for_compare(pd.Series([row[col]])).iloc[0]
                    base_val_norm = normalize_for_compare(pd.Series([base_val])).iloc[0]

                    if row_val_norm != base_val_norm:
                        exception_rows.append({
                            "policy_number": policy_number,
                            "cutoff_month": cutoff_month,
                            "source_table": "policy",
                            "compare_basis": "historical set_month",
                            "history_month": row["set_month"],
                            "inconsistent_field_name": col,
                            "original_input": base_val,
                            "new_input": row[col]
                        })

    policy_exception_df = pd.DataFrame(exception_rows)

    bad_policies = set(policy_exception_df["policy_number"].dropna().astype(str)) if len(policy_exception_df) > 0 else set()

    policy_base = policy_hist.copy()
    policy_base["policy_number_str"] = policy_base["policynumber"].astype("string")

    if bad_policies:
        policy_base = policy_base[~policy_base["policy_number_str"].isin(bad_policies)].copy()

    policy_base = keep_first_by_key(
        policy_base.drop(columns=["policy_number_str"]),
        ["policynumber"],
        sort_cols=["policynumber", "set_month"]
    )

    return policy_base, policy_exception_df

# =========================================================
# BUILD SNAPSHOT
# =========================================================
def build_policy_snapshot(source, cutoff_month):
    seriatim_df = source["seriatim_values"].copy()
    policy_df = source["policy"].copy()
    premiums_df = source["premiums"].copy()
    commission_df = source["commission"].copy()

    # -------------------------
    # Seriatim
    # -------------------------
    seriatim_df = add_missing_cols(seriatim_df, {
        "set_month": None,
        "policynumber": None,
        "totalinitpremium": np.nan,
        "converge": np.nan
    })
    seriatim_df = safe_numeric(seriatim_df, ["totalinitpremium", "converge"])

    seriatim_base = seriatim_df[[
        "set_month",
        "policynumber",
        "totalinitpremium",
        "converge"
    ]].copy()

    seriatim_base = keep_first_by_key(
        seriatim_base,
        ["policynumber"],
        sort_cols=["policynumber", "set_month"]
    )

    # -------------------------
    # Policy with historical consistency check
    # -------------------------
    policy_base, policy_exception_df = build_policy_history_and_exceptions(policy_df, cutoff_month)

    # -------------------------
    # Premiums
    # -------------------------
    premiums_df = add_missing_cols(premiums_df, {
        "policynumber": None,
        "terminateddate": pd.NaT,
        "substatus": None
    })
    premiums_df = safe_datetime(premiums_df, ["terminateddate"])

    premiums_df["termination_type_final"] = premiums_df["substatus"].astype("string")
    premiums_df["termination_type_final"] = premiums_df["termination_type_final"].apply(
        lambda x: x.split(",")[-1].strip() if pd.notna(x) and "," in x else x
    )

    premiums_base = premiums_df[[
        "policynumber",
        "terminateddate",
        "termination_type_final"
    ]].copy()

    premiums_base = premiums_base.rename(columns={
        "terminateddate": "surrender_date"
    })

    premiums_base = keep_first_by_key(premiums_base, ["policynumber"])

    # -------------------------
    # Commission
    # -------------------------
    commission_df = add_missing_cols(commission_df, {
        "policynumber": None,
        "commissionamount": np.nan
    })
    commission_df = safe_numeric(commission_df, ["commissionamount"])

    commission_base = commission_df[[
        "policynumber",
        "commissionamount"
    ]].copy()

    commission_base = commission_base.rename(columns={
        "commissionamount": "initial_commission"
    })

    commission_base = keep_first_by_key(commission_base, ["policynumber"])

    # -------------------------
    # Merge
    # -------------------------
    snapshot_df = (
        seriatim_base
        .merge(policy_base, on="policynumber", how="left")
        .merge(premiums_base, on="policynumber", how="left")
        .merge(commission_base, on="policynumber", how="left")
    )

    snapshot_df["product"] = snapshot_df["policynumber"].apply(derive_product_from_policy)
    snapshot_df["product_group"] = snapshot_df["product"].apply(assign_product_group)

    snapshot_df = snapshot_df.rename(columns={
        "policynumber": "policy_number",
        "term_final": "term",
        "issueyear": "issue_year",
        "issuemonth": "issue_month",
        "issueday": "issue_day",
        "issue_state_final": "issue_state",
        "totalinitpremium": "initial_premium",
        "converge": "quota_share",
        "rateversion": "rate_version",
        "taxqualstatus": "tax_status",
        "ownergender": "gender",
        "joint_annuitant_gender": "annuitant_gender",
        "joint_annuitant_issue_age": "annuitant_issue_age",
        "termination_type_final": "termination_type"
    })

    snapshot_df = add_missing_cols(snapshot_df, {
        "policy_number": None,
        "product": None,
        "term": pd.NA,
        "issue_year": pd.NA,
        "issue_month": pd.NA,
        "issue_day": pd.NA,
        "issue_date": pd.NaT,
        "issue_state": None,
        "initial_premium": np.nan,
        "initial_commission": np.nan,
        "rate_version": pd.NA,
        "gender": None,
        "quota_share": np.nan,
        "tax_status": None,
        "surrender_date": pd.NaT,
        "termination_type": None,
        "annuitant_issue_age": pd.NA,
        "annuitant_gender": None,
        "product_group": None
    })

    snapshot_df = snapshot_df[[
        "policy_number",
        "product",
        "term",
        "issue_year",
        "issue_month",
        "issue_day",
        "issue_date",
        "issue_state",
        "initial_premium",
        "initial_commission",
        "rate_version",
        "gender",
        "quota_share",
        "tax_status",
        "surrender_date",
        "termination_type",
        "annuitant_issue_age",
        "annuitant_gender",
        "product_group"
    ]].copy()

    return snapshot_df, policy_exception_df

# =========================================================
# VALIDATE SNAPSHOT
# =========================================================
def validate_snapshot(snapshot_df):
    df = snapshot_df.copy()

    for col in REQUIRED_COLS:
        df[f"flag_missing_{col}"] = np.where(df[col].isna(), 1, 0)

    flag_cols = [c for c in df.columns if c.startswith("flag_missing_")]
    df["exception_flag"] = df[flag_cols].sum(axis=1)

    invalid_df = df[df["exception_flag"] > 0].copy()
    valid_df = df[df["exception_flag"] == 0].copy()

    return valid_df, invalid_df

# =========================================================
# FIRST LOAD
# =========================================================
def process_group_first_load(group_name, snapshot_all):
    print(f"\nStart first load for {group_name}")

    snapshot_df = snapshot_all.loc[
        snapshot_all["product_group"] == group_name
    ].copy()

    snapshot_df = snapshot_df.drop(columns=["product_group"], errors="ignore")

    valid_snapshot_df, invalid_snapshot_df = validate_snapshot(snapshot_df)

    final_df = keep_first_by_key(
        valid_snapshot_df[TARGET_FINAL_COLS].copy(),
        ["policy_number"]
    )

    # enforce target data types before upload
    final_df = enforce_target_dtypes(final_df)

    upload_to_gbq(final_df, TARGET_TABLES[group_name], if_exists="replace")

    mismatch_df = pd.DataFrame(columns=[
        "policy_number",
        "cutoff_month",
        "product_group",
        "inconsistent_field_name",
        "original_input",
        "new_input"
    ])

    print(f"{group_name}: loaded {len(final_df)} rows")
    return invalid_snapshot_df, mismatch_df

# =========================================================
# MAIN
# =========================================================
def run_fix_process_first_load(cutoff_month):
    print(f"Start fix characteristic first load for historical set_month < {cutoff_month}")

    source = pull_source_tables(cutoff_month)
    snapshot_all, policy_exception_df = build_policy_snapshot(source, cutoff_month)

    snapshot_all = snapshot_all[snapshot_all["product_group"].notna()].copy()

    print(f"Total recognized policy rows in snapshot: {len(snapshot_all)}")
    print(snapshot_all["product_group"].value_counts(dropna=False))

    denali_invalid_df, denali_mismatch_df = process_group_first_load("DENALI", snapshot_all)
    teton_invalid_df, teton_mismatch_df = process_group_first_load("TETON", snapshot_all)

    sheets = {
        "DENALI": pd.concat([denali_invalid_df, denali_mismatch_df], ignore_index=True, sort=False),
        "TETON": pd.concat([teton_invalid_df, teton_mismatch_df], ignore_index=True, sort=False),
        "POLICY_HISTORY_EXCEPTION": policy_exception_df
    }

    write_exception_workbook(
        sheets,
        f"fix_characteristic_exception_before_{cutoff_month}.xlsx"
    )

    print(f"\nFinished first load for historical set_month < {cutoff_month}")

# =========================================================
# RUN
# =========================================================
run_fix_process_first_load(CUT_OFF_MONTH)

Start fix characteristic first load for historical set_month < 202602


C:\Users\m.tang\AppData\Local\Temp\ipykernel_38884\732809864.py:78: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(df[c]):


policy: 7371852 rows
seriatim_values: 6969461 rows
premiums: 81800 rows
commission: 213107 rows



KeyboardInterrupt

